# NB07 - Cross-Notebook Convergence Query

Notebooks NB01 through NB05 each audit a different domain - file quality,
eligibility, accumulators, copay programs, and benefit year rollover. They
run independently and report findings in their own formats. None of them
knows what the others found.

This notebook pulls every finding from every upstream audit into a single
**findings registry** with a common schema, then rolls the data up to
the member level and the plan level. The result is a unified view that
answers the questions the individual notebooks can't:

- Which members have overlapping issues across multiple audit categories?
- Which plans have the broadest exposure?
- Where should the team focus first for the highest remediation ROI?

| Source | What it contributes |
|--------|-------------------|
| NB01 | NULL member IDs, negative amounts, schema drift, freshness SLA, resent files |
| NB02 | Members missing from TPA, termination mismatches, effective date discrepancies |
| NB03 | Deductible and OOPM overcharges (with multi-TPA source aggregation) |
| NB04 | ACCUMULATOR and MAXIMIZER copay errors, state compliance violations |
| NB05 | Rollover failures, cross-year claim misattributions |

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261017_201332


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType
from functools import reduce

spark.sql("USE pbm_sim")

NB = "NB_07"

# Pre-flight -  this notebook re-derives findings from every upstream source,
# so it needs every table that NB01 through NB05 touch.
source_tables = [
    "pbm_sim.plan_design", "pbm_sim.members", "pbm_sim.pbm_eligibility",
    "pbm_sim.tpa_eligibility_feed", "pbm_sim.pbm_accumulators",
    "pbm_sim.tpa_accumulator_feed", "pbm_sim.tpa_accumulator_file_raw",
    "pbm_sim.file_control_log", "pbm_sim.pharmacy_claims",
    "pbm_sim.accumulator_ban_states",
]
for tbl in source_tables:
    validate_table_exists(tbl, NB)

# Load all source tables
df_plan       = spark.table("pbm_sim.plan_design")
df_members    = spark.table("pbm_sim.members")
df_pbm_elig   = spark.table("pbm_sim.pbm_eligibility")
df_tpa_elig   = spark.table("pbm_sim.tpa_eligibility_feed")
df_pbm_accum  = spark.table("pbm_sim.pbm_accumulators")
df_tpa_accum  = spark.table("pbm_sim.tpa_accumulator_feed")
df_raw_file   = spark.table("pbm_sim.tpa_accumulator_file_raw")
df_control    = spark.table("pbm_sim.file_control_log")
df_claims     = spark.table("pbm_sim.pharmacy_claims")

log_entry(NB, "load_source_tables", "SUCCESS",
          detail=f"all {len(source_tables)} source tables loaded")
print("All source tables loaded.")

[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:13:34.386419+00:00", "notebook": "NB_07", "section": "load_source_tables", "status": "SUCCESS", "row_count": 0, "detail": "all 10 source tables loaded", "duration_sec": 0.0}
All source tables loaded.


## Common Finding Schema

Every finding, regardless of source notebook, gets projected into this
structure. The `issue_category` maps to the notebook that originally found
it, `issue_type` is the specific check within that notebook, and `severity`
is assigned based on operational impact and dollar exposure.

| Field | Description |
|-------|-------------|
| `member_id` | Affected member (NULL for file-level findings) |
| `plan_id` | Associated plan |
| `issue_category` | FILE_QUALITY, ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER |
| `issue_type` | Specific finding (e.g. DEDUCTIBLE_OVERCHARGE, MAXIMIZER_PRE_EXHAUSTION_ERROR) |
| `severity` | CRITICAL, HIGH, MEDIUM, LOW |
| `dollar_exposure` | Estimated financial impact (0.0 if non-monetary) |
| `detail` | Human-readable description of the finding |

In [0]:
# Define the schema for the finding table.
# Using a fixed StructType so all finding DataFrames can be unioned
# cleanly regardless of which checks produced them.
finding_schema = StructType([
    StructField("member_id", StringType(), True),
    StructField("plan_id", StringType(), True),
    StructField("issue_category", StringType(), False),
    StructField("issue_type", StringType(), False),
    StructField("severity", StringType(), False),
    StructField("dollar_exposure", DoubleType(), False),
    StructField("detail", StringType(), False),
])

print("Finding schema defined.")

Finding schema defined.


## NB01 Findings - File Quality

Re-derive the inbound file validation checks: NULL member IDs and
negative accumulator amounts. These are record-level data quality issues
that indicate problems in the TPA's export process before the data even
enters reconciliation.

In [0]:
# NULL member_id records - these can't be matched to any member,
# so they drop out of every downstream reconciliation silently.
df_with_id = df_raw_file.filter(F.col("member_id").isNotNull())

df_nulls = (
    df_raw_file
    .filter(F.col("member_id").isNull())
    .select(
        F.lit(None).cast("string").alias("member_id"),
        F.col("plan_id"),
        F.lit("FILE_QUALITY").alias("issue_category"),
        F.lit("NULL_MEMBER_ID").alias("issue_type"),
        F.lit("HIGH").alias("severity"),
        F.lit(0.0).alias("dollar_exposure"),
        F.lit("Record in raw TPA file has no member_id - cannot be matched or applied").alias("detail"),
    )
)

# Negative accumulator amounts - a deductible or OOPM applied value
# should never be negative. This usually means a reversal or adjustment
# that wasn't handled correctly in the TPA's export.
df_neg = (
    df_with_id
    .filter(
        (F.col("medical_deductible_applied") < 0)
        | (F.col("medical_oopm_applied") < 0)
    )
    .select(
        F.col("member_id"),
        F.col("plan_id"),
        F.lit("FILE_QUALITY").alias("issue_category"),
        F.lit("NEGATIVE_AMOUNT").alias("issue_type"),
        F.lit("MEDIUM").alias("severity"),
        F.lit(0.0).alias("dollar_exposure"),
        F.concat(
            F.lit("Negative accumulator value: ded="),
            F.round("medical_deductible_applied", 2).cast("string"),
            F.lit(", oopm="),
            F.round("medical_oopm_applied", 2).cast("string"),
        ).alias("detail"),
    )
)

nb01_count = df_nulls.count() + df_neg.count()
print(f"NB01 findings: {df_nulls.count()} NULL member_ids, {df_neg.count()} negative amounts : {nb01_count} total")

NB01 findings: 20 NULL member_ids, 10 negative amounts : 30 total


### NB01 Extended - File Exchange Quality

Schema drift detection, file freshness SLA violations, and resent file
detection. These are file-level findings that flag problems in the TPA
delivery pipeline itself, not in individual records.

In [0]:
# Schema drift detection.
# Compare actual raw file columns against the expected schema.
# A new column means the TPA changed their export without telling us.
# A missing column means something broke in their extract.
EXPECTED_COLUMNS = {
    "member_id": "string", "plan_id": "string",
    "medical_deductible_applied": "double", "medical_oopm_applied": "double",
    "benefit_year": "int", "file_date": "date",
}

actual_cols = {f.name: f.dataType.simpleString() for f in df_raw_file.schema.fields}
expected_set = set(EXPECTED_COLUMNS.keys())
actual_set = set(actual_cols.keys())

schema_drift_rows = []
for col_name in actual_set - expected_set:
    schema_drift_rows.append(("SCHEMA_DRIFT", f"Unexpected column '{col_name}' ({actual_cols[col_name]}) in raw file - not in expected schema"))
for col_name in expected_set - actual_set:
    schema_drift_rows.append(("SCHEMA_DRIFT", f"Expected column '{col_name}' missing from raw file"))
for col_name in expected_set & actual_set:
    if actual_cols[col_name] != EXPECTED_COLUMNS[col_name]:
        schema_drift_rows.append(("SCHEMA_DRIFT", f"Column '{col_name}' type changed: expected {EXPECTED_COLUMNS[col_name]}, got {actual_cols[col_name]}"))

if schema_drift_rows:
    df_schema_drift = spark.createDataFrame(
        [(None, None, "FILE_QUALITY", st, "MEDIUM", 0.0, detail) for st, detail in schema_drift_rows],
        schema=finding_schema,
    )
else:
    df_schema_drift = spark.createDataFrame([], schema=finding_schema)

# File freshness SLA monitoring.
# If the file date is older than yesterday, the TPA may have missed
# a delivery or sent a stale file.
from datetime import date, timedelta

today = date.today()
yesterday = today - timedelta(days=1)

file_dates = [row["file_date"] for row in df_raw_file.select("file_date").distinct().collect()]
freshness_rows = []

if len(file_dates) > 1:
    dates_str = ", ".join(str(d) for d in sorted(file_dates))
    freshness_rows.append(("CONCATENATED_FILE", f"Raw file contains {len(file_dates)} distinct file_dates ({dates_str}) - possible file concatenation"))

for fd in file_dates:
    if fd is not None and fd < yesterday:
        freshness_rows.append(("STALE_FILE", f"File dated {fd} is older than SLA window (expected {yesterday} or later)"))

if freshness_rows:
    df_freshness = spark.createDataFrame(
        [(None, None, "FILE_QUALITY", st, "HIGH" if st == "STALE_FILE" else "MEDIUM", 0.0, detail) for st, detail in freshness_rows],
        schema=finding_schema,
    )
else:
    df_freshness = spark.createDataFrame([], schema=finding_schema)

# Resent file detection via control log.
# Multiple entries for the same file_date means the TPA sent it more
# than once - could be a correction or could be a duplicate.
if "file_date" in df_control.columns:
    df_resent_check = (
        df_control
        .groupBy("file_date")
        .agg(F.count("*").alias("delivery_count"))
        .filter(F.col("delivery_count") > 1)
    )
    resent_rows = [
        (None, None, "FILE_QUALITY", "RESENT_FILE", "MEDIUM", 0.0,
         f"File date {row['file_date']} has {row['delivery_count']} entries in control log - file was resent")
        for row in df_resent_check.collect()
    ]
    df_resent = spark.createDataFrame(resent_rows, schema=finding_schema) if resent_rows else spark.createDataFrame([], schema=finding_schema)
else:
    df_resent = spark.createDataFrame([], schema=finding_schema)

nb01_ext_count = df_schema_drift.count() + df_freshness.count() + df_resent.count()
print(f"NB01 extended: {df_schema_drift.count()} schema drift, "
      f"{df_freshness.count()} freshness SLA, {df_resent.count()} resent files : {nb01_ext_count} total")

log_entry(NB, "nb01_findings", "SUCCESS",
          detail=f"core={nb01_count}, extended={nb01_ext_count}")

NB01 extended: 1 schema drift, 1 freshness SLA, 0 resent files : 2 total
[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:13:46.384097+00:00", "notebook": "NB_07", "section": "nb01_findings", "status": "SUCCESS", "row_count": 0, "detail": "core=30, extended=2", "duration_sec": 0.0}


## NB02 Findings - Eligibility

Re-derive the eligibility reconciliation checks: members in PBM but
missing from the TPA feed (no medical accumulator data flowing),
termination status mismatches (active in PBM but termed per TPA),
and effective date discrepancies between systems.

In [0]:
# Members in PBM but missing from TPA.
# These members have pharmacy coverage but no medical accumulator
# data is flowing - the combined deductible check in NB03 can't
# work for them.
df_pbm_ids = df_pbm_elig.select("member_id").distinct()
df_tpa_ids = df_tpa_elig.select("member_id").distinct()

df_missing_tpa = (
    df_pbm_ids
    .join(df_tpa_ids, on="member_id", how="left_anti")
    .join(df_pbm_elig.select("member_id", "plan_id"), on="member_id")
    .select(
        "member_id", "plan_id",
        F.lit("ELIGIBILITY").alias("issue_category"),
        F.lit("MISSING_FROM_TPA").alias("issue_type"),
        F.lit("HIGH").alias("severity"),
        F.lit(0.0).alias("dollar_exposure"),
        F.lit("Member exists in PBM eligibility but is absent from TPA feed - no medical accumulator data").alias("detail"),
    )
)

# Termination mismatches.
# Active in PBM (term_date is null) but terminated per TPA (term_date is not null).
# These members are getting claims adjudicated as active when the TPA
# thinks they're terminated - potential post-termination exposure.
df_elig_joined = (
    df_pbm_elig.alias("pbm")
    .join(df_tpa_elig.alias("tpa"), on="member_id", how="inner")
    .select(
        F.col("pbm.member_id"),
        F.col("pbm.plan_id"),
        F.col("pbm.term_date").alias("pbm_term_date"),
        F.col("tpa.term_date").alias("tpa_term_date"),
        F.col("pbm.effective_date").alias("pbm_effective_date"),
        F.col("tpa.effective_date").alias("tpa_effective_date"),
    )
)

df_term_mismatch = (
    df_elig_joined
    .filter(
        F.col("pbm_term_date").isNull() & F.col("tpa_term_date").isNotNull()
    )
    .select(
        "member_id", "plan_id",
        F.lit("ELIGIBILITY").alias("issue_category"),
        F.lit("TERM_MISMATCH").alias("issue_type"),
        F.lit("CRITICAL").alias("severity"),
        F.lit(0.0).alias("dollar_exposure"),
        F.concat(
            F.lit("Active in PBM but terminated per TPA on "),
            F.col("tpa_term_date").cast("string"),
        ).alias("detail"),
    )
)

# Effective date mismatches
df_eff_mismatch = (
    df_elig_joined
    .filter(F.col("pbm_effective_date") != F.col("tpa_effective_date"))
    .select(
        "member_id", "plan_id",
        F.lit("ELIGIBILITY").alias("issue_category"),
        F.lit("EFFECTIVE_DATE_MISMATCH").alias("issue_type"),
        F.lit("MEDIUM").alias("severity"),
        F.lit(0.0).alias("dollar_exposure"),
        F.concat(
            F.lit("PBM effective="),
            F.col("pbm_effective_date").cast("string"),
            F.lit(" vs TPA effective="),
            F.col("tpa_effective_date").cast("string"),
            F.lit(" ("),
            F.abs(F.datediff("pbm_effective_date", "tpa_effective_date")).cast("string"),
            F.lit(" day difference)"),
        ).alias("detail"),
    )
)

nb02_count = df_missing_tpa.count() + df_term_mismatch.count() + df_eff_mismatch.count()
print(f"NB02 findings: {df_missing_tpa.count()} missing from TPA, "
      f"{df_term_mismatch.count()} term mismatches, "
      f"{df_eff_mismatch.count()} effective date mismatches : {nb02_count} total")

log_entry(NB, "nb02_findings", "SUCCESS",
          detail=f"missing_tpa={df_missing_tpa.count()}, term={df_term_mismatch.count()}, eff_date={df_eff_mismatch.count()}")

NB02 findings: 125 missing from TPA, 500 term mismatches, 200 effective date mismatches : 825 total
[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:13:56.634543+00:00", "notebook": "NB_07", "section": "nb02_findings", "status": "SUCCESS", "row_count": 0, "detail": "missing_tpa=125, term=500, eff_date=200", "duration_sec": 0.0}


## NB03 Findings - Accumulator Overcharges

Re-derive the combined deductible and OOPM overcharges by joining
pharmacy (PBM) and medical (TPA) accumulators against plan design limits.
If multi-TPA carve-out arrangements exist (indicated by a `tpa_source`
column), we aggregate across TPA sources first so that split medical
accumulators are summed before comparison against plan limits.

In [0]:
df_pbm_a = df_pbm_accum.filter(F.col("benefit_year") == 2026)
df_tpa_a = df_tpa_accum.filter(F.col("benefit_year") == 2026)

# Multi-TPA aggregation: if tpa_source column exists, sum across TPA
# sources before comparing. A member with split medical coverage across
# two TPAs might look fine on each side individually but exceed the
# plan limit when combined.
if "tpa_source" in df_tpa_a.columns:
    df_tpa_a = (
        df_tpa_a
        .groupBy("member_id", "plan_id", "benefit_year")
        .agg(
            F.sum("medical_deductible_applied").alias("medical_deductible_applied"),
            F.sum("medical_oopm_applied").alias("medical_oopm_applied"),
            F.max("last_claim_date").alias("last_claim_date"),
            F.countDistinct("tpa_source").alias("tpa_source_count"),
        )
    )

df_combined = (
    df_pbm_a.alias("pbm")
    .join(df_tpa_a.alias("tpa"), on=["member_id", "plan_id"], how="inner")
    .join(df_plan, on="plan_id")
    .withColumn(
        "combined_deductible",
        F.round(F.col("pbm.pharmacy_deductible_applied") + F.col("tpa.medical_deductible_applied"), 2),
    )
    .withColumn(
        "combined_oopm",
        F.round(F.col("pbm.pharmacy_oopm_applied") + F.col("tpa.medical_oopm_applied"), 2),
    )
    .withColumn(
        "ded_overage",
        F.greatest(F.col("combined_deductible") - F.col("individual_deductible"), F.lit(0)),
    )
    .withColumn(
        "oopm_overage",
        F.greatest(F.col("combined_oopm") - F.col("individual_oopm"), F.lit(0)),
    )
)

# Deductible overcharges
df_ded_over = (
    df_combined
    .filter(F.col("ded_overage") > 0)
    .select(
        "member_id", "plan_id",
        F.lit("ACCUMULATOR").alias("issue_category"),
        F.lit("DEDUCTIBLE_OVERCHARGE").alias("issue_type"),
        F.lit("CRITICAL").alias("severity"),
        F.round("ded_overage", 2).alias("dollar_exposure"),
        F.concat(
            F.lit("Combined deductible $"),
            F.round("combined_deductible", 2).cast("string"),
            F.lit(" exceeds plan limit $"),
            F.col("individual_deductible").cast("string"),
            F.lit(" (overage $"),
            F.round("ded_overage", 2).cast("string"),
            F.lit(")"),
        ).alias("detail"),
    )
)

# OOPM overcharges
df_oopm_over = (
    df_combined
    .filter(F.col("oopm_overage") > 0)
    .select(
        "member_id", "plan_id",
        F.lit("ACCUMULATOR").alias("issue_category"),
        F.lit("OOPM_OVERCHARGE").alias("issue_type"),
        F.lit("CRITICAL").alias("severity"),
        F.round("oopm_overage", 2).alias("dollar_exposure"),
        F.concat(
            F.lit("Combined OOPM $"),
            F.round("combined_oopm", 2).cast("string"),
            F.lit(" exceeds plan limit $"),
            F.col("individual_oopm").cast("string"),
            F.lit(" (overage $"),
            F.round("oopm_overage", 2).cast("string"),
            F.lit(")"),
        ).alias("detail"),
    )
)

nb03_count = df_ded_over.count() + df_oopm_over.count()
ded_total = df_ded_over.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
oopm_total = df_oopm_over.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
print(f"NB03 findings: {df_ded_over.count()} deductible overcharges (${ded_total:,.2f}), "
      f"{df_oopm_over.count()} OOPM overcharges (${oopm_total:,.2f}) : {nb03_count} total")

log_entry(NB, "nb03_findings", "SUCCESS",
          detail=f"ded_overcharges={df_ded_over.count()} (${ded_total:,.2f}), oopm_overcharges={df_oopm_over.count()} (${oopm_total:,.2f})")

NB03 findings: 1674 deductible overcharges ($569,698.38), 375 OOPM overcharges ($171,378.25) : 2049 total
[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:14:08.937171+00:00", "notebook": "NB_07", "section": "nb03_findings", "status": "SUCCESS", "row_count": 0, "detail": "ded_overcharges=1674 ($569,698.38), oopm_overcharges=375 ($171,378.25)", "duration_sec": 0.0}


## NB04 Findings - Copay Accumulator Program

Re-derive copay accumulator errors for ACCUMULATOR plans (manufacturer
copay incorrectly credited toward deductible) and MAXIMIZER plans (phase
transition errors when the copay card runs out). Also derives state
compliance violations using the managed reference table
`pbm_sim.accumulator_ban_states` with program-type-aware ban flags.

In [0]:
from pyspark.sql.window import Window

# --- ACCUMULATOR copay errors ---
# On ACCUMULATOR plans, manufacturer copay should NOT count toward
# the member's deductible. Any claim where it does is an error.
df_coupon_claims = df_claims.filter(F.col("manufacturer_copay_amount") > 0)

df_copay_errors = (
    df_coupon_claims
    .join(df_plan, on="plan_id")
    .filter(
        (F.col("accumulator_program_type") == "ACCUMULATOR")
        & (F.col("manufacturer_copay_counted_toward_deductible") == "Y")
    )
    .select(
        "member_id", "plan_id",
        F.lit("COPAY_PROGRAM").alias("issue_category"),
        F.lit("COUPON_INCORRECTLY_CREDITED").alias("issue_type"),
        F.lit("HIGH").alias("severity"),
        F.round("manufacturer_copay_amount", 2).alias("dollar_exposure"),
        F.concat(
            F.lit("Claim "),
            F.col("claim_id"),
            F.lit(": $"),
            F.round("manufacturer_copay_amount", 2).cast("string"),
            F.lit(" manufacturer copay credited toward deductible on ACCUMULATOR plan ("),
            F.col("drug_name"),
            F.lit(")"),
        ).alias("detail"),
    )
)

# --- MAXIMIZER copay card exhaustion errors ---
# Detect phase transitions: once cumulative manufacturer_copay_amount stops
# increasing (card exhausted), member OOP should count toward accumulators.
# The cumulative window function tracks the running total per member/drug/year.

df_maximizer_claims = (
    df_coupon_claims
    .join(df_plan, on="plan_id")
    .filter(F.col("accumulator_program_type") == "MAXIMIZER")
)

w_cumulative = (
    Window
    .partitionBy("member_id", "drug_name", "benefit_year")
    .orderBy("service_date")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

df_maximizer_phased = (
    df_maximizer_claims
    .withColumn("cumulative_copay", F.sum("manufacturer_copay_amount").over(w_cumulative))
    .withColumn(
        "copay_phase",
        F.when(F.col("manufacturer_copay_amount") > 0, "PRE_EXHAUSTION")
        .when(
            (F.col("manufacturer_copay_amount") == 0) & (F.col("cumulative_copay") > 0),
            "POST_EXHAUSTION",
        )
        .otherwise("NO_COPAY_CARD"),
    )
)

# Pre-exhaustion errors: manufacturer $ credited toward deductible when it shouldn't be
df_pre_exhaust_errors = (
    df_maximizer_phased
    .filter(
        (F.col("copay_phase") == "PRE_EXHAUSTION")
        & (F.col("manufacturer_copay_counted_toward_deductible") == "Y")
    )
    .select(
        "member_id", "plan_id",
        F.lit("COPAY_PROGRAM").alias("issue_category"),
        F.lit("MAXIMIZER_PRE_EXHAUSTION_ERROR").alias("issue_type"),
        F.lit("HIGH").alias("severity"),
        F.round("manufacturer_copay_amount", 2).alias("dollar_exposure"),
        F.concat(
            F.lit("MAXIMIZER pre-exhaustion: $"),
            F.round("manufacturer_copay_amount", 2).cast("string"),
            F.lit(" manufacturer copay credited toward deductible for "),
            F.col("drug_name"),
            F.lit(" (card still active - should not count)"),
        ).alias("detail"),
    )
)

# Post-exhaustion errors: member OOP not credited after card exhaustion.
# Once the copay card is empty, the member is paying out of pocket
# and those payments SHOULD count toward their accumulators.
df_post_exhaust_errors = (
    df_maximizer_phased
    .filter(
        (F.col("copay_phase") == "POST_EXHAUSTION")
        & (F.col("member_deductible_applied") > 0)
        & (F.col("manufacturer_copay_counted_toward_deductible") == "N")
    )
    .select(
        "member_id", "plan_id",
        F.lit("COPAY_PROGRAM").alias("issue_category"),
        F.lit("MAXIMIZER_POST_EXHAUSTION_ERROR").alias("issue_type"),
        F.lit("HIGH").alias("severity"),
        F.round("member_deductible_applied", 2).alias("dollar_exposure"),
        F.concat(
            F.lit("MAXIMIZER post-exhaustion: $"),
            F.round("member_deductible_applied", 2).cast("string"),
            F.lit(" member OOP not credited toward deductible for "),
            F.col("drug_name"),
            F.lit(" (copay card exhausted - member OOP should count)"),
        ).alias("detail"),
    )
)

# --- State compliance violations (reference table) ---
# Some states ban accumulator or maximizer programs for fully-insured plans.
# Self-funded (ERISA) plans are exempt from state law - federal preemption.
df_ban_states = spark.table("pbm_sim.accumulator_ban_states")

df_violating_plans = (
    df_plan
    .filter(F.col("funding_model") == "FULLY_INSURED")
    .join(df_ban_states, df_plan["state_domicile"] == df_ban_states["state_code"])
    .filter(
        (
            (F.col("accumulator_program_type") == "ACCUMULATOR")
            & (F.col("bans_accumulator") == True)
        )
        | (
            (F.col("accumulator_program_type") == "MAXIMIZER")
            & (F.col("bans_maximizer") == True)
        )
    )
    .filter(~F.col("erisa_exempt"))
)

# Create a finding for each member on a violating plan
df_compliance = (
    df_members
    .join(df_violating_plans.select("plan_id", "state_domicile", "accumulator_program_type"), on="plan_id")
    .select(
        "member_id", "plan_id",
        F.lit("COPAY_PROGRAM").alias("issue_category"),
        F.lit("STATE_LAW_VIOLATION").alias("issue_type"),
        F.lit("CRITICAL").alias("severity"),
        F.lit(0.0).alias("dollar_exposure"),
        F.concat(
            F.lit("Plan uses "),
            F.col("accumulator_program_type"),
            F.lit(" program but is FULLY_INSURED in "),
            F.col("state_domicile"),
            F.lit(" (ban state) - configuration violates state law"),
        ).alias("detail"),
    )
    .distinct()
)

nb04_count = (df_copay_errors.count() + df_pre_exhaust_errors.count()
              + df_post_exhaust_errors.count() + df_compliance.count())
copay_dollars = df_copay_errors.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
pre_exhaust_dollars = df_pre_exhaust_errors.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
post_exhaust_dollars = df_post_exhaust_errors.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
print(f"NB04 findings: {df_copay_errors.count()} ACCUMULATOR copay errors (${copay_dollars:,.2f}), "
      f"{df_pre_exhaust_errors.count()} MAXIMIZER pre-exhaustion (${pre_exhaust_dollars:,.2f}), "
      f"{df_post_exhaust_errors.count()} MAXIMIZER post-exhaustion (${post_exhaust_dollars:,.2f}), "
      f"{df_compliance.count()} state compliance : {nb04_count} total")

log_entry(NB, "nb04_findings", "SUCCESS",
          detail=f"copay_errors={df_copay_errors.count()}, pre_exhaust={df_pre_exhaust_errors.count()}, "
                  f"post_exhaust={df_post_exhaust_errors.count()}, state_compliance={df_compliance.count()}")

NB04 findings: 60 ACCUMULATOR copay errors ($51,082.96), 3848 MAXIMIZER pre-exhaustion ($3,000,498.97), 0 MAXIMIZER post-exhaustion ($0.00), 0 state compliance : 3908 total
[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:14:24.677456+00:00", "notebook": "NB_07", "section": "nb04_findings", "status": "SUCCESS", "row_count": 0, "detail": "copay_errors=60, pre_exhaust=3848, post_exhaust=0, state_compliance=0", "duration_sec": 0.0}


## NB05 Findings - Benefit Year Rollover

Re-derive rollover failures (active members with 2025 accumulator records
but no 2026 record initialized) and cross-year claim misattributions
(service date in 2025, processing date in 2026, incorrectly tagged as
benefit year 2026).

In [0]:
# Rollover failures.
# Members who had a 2025 accumulator record but no 2026 record,
# excluding those legitimately terminated before Jan 1 2026.
df_accum_2025 = df_pbm_accum.filter(F.col("benefit_year") == 2025)
df_accum_2026 = df_pbm_accum.filter(F.col("benefit_year") == 2026)

df_had_2025 = df_accum_2025.select("member_id", "plan_id").distinct()
df_has_2026 = df_accum_2026.select("member_id").distinct()

df_missing_2026 = df_had_2025.join(df_has_2026, on="member_id", how="left_anti")

# Filter out legitimately terminated members
df_rollover_raw = (
    df_missing_2026
    .join(df_members.select("member_id", "term_date"), on="member_id")
    .filter(
        F.col("term_date").isNull() | (F.col("term_date") >= F.lit("2026-01-01"))
    )
)

df_rollover = (
    df_rollover_raw
    .select(
        "member_id", "plan_id",
        F.lit("ROLLOVER").alias("issue_category"),
        F.lit("ROLLOVER_FAILURE").alias("issue_type"),
        F.lit("HIGH").alias("severity"),
        F.lit(0.0).alias("dollar_exposure"),
        F.lit("Active member has 2025 accumulator record but no 2026 record - rollover job skipped").alias("detail"),
    )
)

# Cross-year claim misattributions.
# These are claims with a 2025 service date but tagged as benefit
# year 2026 because the processing date fell in January.
df_cross_year = (
    df_claims
    .filter(
        (F.year(F.col("service_date")) == 2025)
        & (F.year(F.col("processing_date")) == 2026)
        & (F.col("benefit_year") == 2026)  # Incorrectly attributed
    )
    .select(
        "member_id", "plan_id",
        F.lit("ROLLOVER").alias("issue_category"),
        F.lit("CROSS_YEAR_MISATTRIBUTION").alias("issue_type"),
        F.lit("MEDIUM").alias("severity"),
        F.round("member_deductible_applied", 2).alias("dollar_exposure"),
        F.concat(
            F.lit("Claim service_date "),
            F.col("service_date").cast("string"),
            F.lit(" (2025) but tagged as benefit_year 2026 - deductible of $"),
            F.round("member_deductible_applied", 2).cast("string"),
            F.lit(" counted against wrong year"),
        ).alias("detail"),
    )
)

nb05_count = df_rollover.count() + df_cross_year.count()
cross_year_dollars = df_cross_year.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
print(f"NB05 findings: {df_rollover.count()} rollover failures, "
      f"{df_cross_year.count()} cross-year misattributions (${cross_year_dollars:,.2f}) : {nb05_count} total")

log_entry(NB, "nb05_findings", "SUCCESS",
          detail=f"rollover={df_rollover.count()}, cross_year={df_cross_year.count()}")

NB05 findings: 375 rollover failures, 40 cross-year misattributions ($365.40) : 415 total
[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:14:34.235594+00:00", "notebook": "NB_07", "section": "nb05_findings", "status": "SUCCESS", "row_count": 0, "detail": "rollover=375, cross_year=40", "duration_sec": 0.0}


## Unified Findings Registry

Union all individual finding DataFrames into a single table. This becomes
the canonical source for every downstream rollup and the executive dashboard.


In [0]:
# Union all finding DataFrames into a single registry.
# Using unionByName so column order doesn't matter across the
# different source DataFrames.
all_findings = [
    df_nulls, df_neg,                       # NB01 - core
    df_schema_drift, df_freshness,          # NB01 - file exchange
    df_resent,
    df_missing_tpa, df_term_mismatch,       # NB02
    df_eff_mismatch,
    df_ded_over, df_oopm_over,              # NB03
    df_copay_errors,                        # NB04 - ACCUMULATOR
    df_pre_exhaust_errors,                  # NB04 - MAXIMIZER
    df_post_exhaust_errors,
    df_compliance,                          # NB04 - state compliance
    df_rollover, df_cross_year,             # NB05
]

df_registry = reduce(lambda a, b: a.unionByName(b), all_findings)

# Write to Delta
(
    df_registry.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("pbm_sim.findings_registry")
)

total = df_registry.count()
print(f"findings_registry written: {total} total findings")
print()

# Breakdown by category
print("Findings by category:")
display(
    df_registry
    .groupBy("issue_category")
    .agg(
        F.count("*").alias("findings"),
        F.countDistinct("issue_type").alias("issue_types"),
        F.round(F.sum("dollar_exposure"), 2).alias("total_dollar_exposure"),
    )
    .orderBy(F.col("findings").desc())
)

log_entry(NB, "findings_registry", "SUCCESS",
          detail=f"total_findings={total}")

findings_registry written: 7229 total findings

Findings by category:


issue_category,findings,issue_types,total_dollar_exposure
COPAY_PROGRAM,3908,2,3051581.93
ACCUMULATOR,2049,2,741076.63
ELIGIBILITY,825,3,0.0
ROLLOVER,415,2,365.4
FILE_QUALITY,32,4,0.0


[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:14:59.983776+00:00", "notebook": "NB_07", "section": "findings_registry", "status": "SUCCESS", "row_count": 0, "detail": "total_findings=7229", "duration_sec": 0.0}


## Member-Level Convergence

This is where the cross-notebook value shows up. For each member, we count
how many distinct issue types affect them, which categories they span,
and their total dollar exposure. Members who appear in multiple categories
are the ones whose data is comprehensively broken - fixing one issue
without seeing the others leads to incomplete remediation.

The `overlap_count` column tracks how many distinct audit categories
flagged the member. A member with `overlap_count = 3` has findings in
three of the five domains (e.g., eligibility + accumulator + copay).
These are the highest-priority cases for coordinated investigation.

In [0]:
# Only member-level findings (exclude file-level NULLs)
df_member_findings = df_registry.filter(F.col("member_id").isNotNull())

# Severity rank for picking the worst severity per member.
# CRITICAL=1 so min() gives us the most severe.
severity_rank = F.when(F.col("severity") == "CRITICAL", 1) \
    .when(F.col("severity") == "HIGH", 2) \
    .when(F.col("severity") == "MEDIUM", 3) \
    .otherwise(4)

df_member_rollup = (
    df_member_findings
    .groupBy("member_id")
    .agg(
        F.first("plan_id").alias("plan_id"),
        F.countDistinct("issue_type").alias("distinct_issue_types"),
        F.countDistinct("issue_category").alias("overlap_count"),
        F.collect_set("issue_category").alias("categories"),
        F.round(F.sum("dollar_exposure"), 2).alias("total_dollar_exposure"),
        F.count("*").alias("total_findings"),
        F.min(severity_rank).alias("worst_severity_rank"),
    )
    .withColumn(
        "worst_severity",
        F.when(F.col("worst_severity_rank") == 1, "CRITICAL")
        .when(F.col("worst_severity_rank") == 2, "HIGH")
        .when(F.col("worst_severity_rank") == 3, "MEDIUM")
        .otherwise("LOW"),
    )
    .drop("worst_severity_rank")
    .orderBy(F.col("overlap_count").desc(), F.col("total_dollar_exposure").desc())
)

(
    df_member_rollup.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("pbm_sim.findings_member_rollup")
)

total_members = df_member_rollup.count()
multi_category = df_member_rollup.filter(F.col("overlap_count") > 1).count()

print(f"findings_member_rollup written: {total_members} members with at least one finding")
print(f"Members with findings in multiple categories: {multi_category}")
print()

print("Top 20 members by overlap (multi-category issues):")
display(df_member_rollup.limit(20))

log_entry(NB, "member_rollup", "SUCCESS",
          detail=f"total_members={total_members}, multi_category={multi_category}")

findings_member_rollup written: 5944 members with at least one finding
Members with findings in multiple categories: 176

Top 20 members by overlap (multi-category issues):


member_id,plan_id,distinct_issue_types,overlap_count,categories,total_dollar_exposure,total_findings,worst_severity
MBR1027681,PLN016,3,3,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM)",2202.73,3,CRITICAL
MBR1047754,PLN039,3,3,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM)",877.53,3,CRITICAL
MBR1043223,PLN032,2,2,"List(ACCUMULATOR, COPAY_PROGRAM)",4610.6,5,CRITICAL
MBR1043155,PLN032,2,2,"List(ACCUMULATOR, COPAY_PROGRAM)",4581.51,4,CRITICAL
MBR1037410,PLN026,2,2,"List(ACCUMULATOR, COPAY_PROGRAM)",4289.02,3,CRITICAL
MBR1012687,PLN006,2,2,"List(ACCUMULATOR, COPAY_PROGRAM)",3882.1,4,CRITICAL
MBR1011963,PLN006,2,2,"List(ACCUMULATOR, COPAY_PROGRAM)",3686.04,3,CRITICAL
MBR1048151,PLN039,2,2,"List(ACCUMULATOR, COPAY_PROGRAM)",3318.28,4,CRITICAL
MBR1012226,PLN006,2,2,"List(ACCUMULATOR, COPAY_PROGRAM)",3247.66,4,CRITICAL
MBR1012427,PLN006,2,2,"List(ACCUMULATOR, COPAY_PROGRAM)",3195.67,4,CRITICAL


[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:15:32.709739+00:00", "notebook": "NB_07", "section": "member_rollup", "status": "SUCCESS", "row_count": 0, "detail": "total_members=5944, multi_category=176", "duration_sec": 0.0}


## Overlap Analysis

How much overlap exists across audit categories? This tells us whether
the notebooks are finding isolated issues or whether the same members
keep appearing across multiple domains. High overlap suggests a systemic
upstream problem rather than independent data quality issues.

In [0]:
print("Distribution of overlap counts (number of audit categories per member):")
display(
    df_member_rollup
    .groupBy("overlap_count")
    .agg(
        F.count("*").alias("members"),
        F.round(F.sum("total_dollar_exposure"), 2).alias("combined_exposure"),
    )
    .orderBy("overlap_count")
)

print()
print("Category co-occurrence (members appearing in multiple categories):")
# Explode categories and self-join to find pairs.
# This shows which audit domains tend to co-occur on the same members -
# e.g., if ELIGIBILITY and ACCUMULATOR always overlap, there's likely
# a shared root cause in the enrollment feed.
df_exploded = (
    df_member_rollup
    .filter(F.col("overlap_count") > 1)
    .select("member_id", F.explode("categories").alias("category"))
)

df_pairs = (
    df_exploded.alias("a")
    .join(df_exploded.alias("b"), on="member_id")
    .filter(F.col("a.category") < F.col("b.category"))
    .groupBy(F.col("a.category").alias("category_1"), F.col("b.category").alias("category_2"))
    .agg(F.countDistinct("member_id").alias("shared_members"))
    .orderBy(F.col("shared_members").desc())
)

display(df_pairs)

log_entry(NB, "overlap_analysis", "SUCCESS",
          detail=f"multi_category_members={multi_category}")

Distribution of overlap counts (number of audit categories per member):


overlap_count,members,combined_exposure
1,5768,3557986.47
2,174,231957.23
3,2,3080.26



Category co-occurrence (members appearing in multiple categories):


category_1,category_2,shared_members
ACCUMULATOR,COPAY_PROGRAM,137
ACCUMULATOR,ELIGIBILITY,17
COPAY_PROGRAM,ELIGIBILITY,14
ACCUMULATOR,ROLLOVER,6
ELIGIBILITY,ROLLOVER,3
COPAY_PROGRAM,ROLLOVER,2
ELIGIBILITY,FILE_QUALITY,1


[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:15:48.954759+00:00", "notebook": "NB_07", "section": "overlap_analysis", "status": "SUCCESS", "row_count": 0, "detail": "multi_category_members=176", "duration_sec": 0.0}


## Plan-Level Rollup

For account management - which employer groups have the broadest and
deepest exposure? This rollup helps prioritize plan-level remediation
conversations and identifies plans that may have systemic configuration
issues rather than isolated member-level data problems.

In [0]:
df_plan_rollup = (
    df_member_findings
    .groupBy("plan_id")
    .agg(
        F.countDistinct("member_id").alias("affected_members"),
        F.countDistinct("issue_category").alias("categories_affected"),
        F.countDistinct("issue_type").alias("distinct_issue_types"),
        F.round(F.sum("dollar_exposure"), 2).alias("total_dollar_exposure"),
        F.count("*").alias("total_findings"),
        F.collect_set("issue_category").alias("categories"),
    )
    .join(
        df_plan.select("plan_id", "plan_name", "funding_model", "tpa_name"),
        on="plan_id",
    )
    .orderBy(F.col("total_dollar_exposure").desc())
)

(
    df_plan_rollup.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("pbm_sim.findings_plan_rollup")
)

print("findings_plan_rollup written:")
display(df_plan_rollup)

log_entry(NB, "plan_rollup", "SUCCESS",
          detail=f"plans={df_plan_rollup.count()}")

findings_plan_rollup written:


plan_id,affected_members,categories_affected,distinct_issue_types,total_dollar_exposure,total_findings,categories,plan_name,funding_model,tpa_name
PLN006,841,4,8,857972.98,1183,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Zeta Manufacturing,SELF_FUNDED,TPA 4
PLN016,579,5,9,591875.95,815,"List(FILE_QUALITY, ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Pi Aerospace Defense,SELF_FUNDED,TPA 5
PLN026,476,5,8,514111.22,677,"List(FILE_QUALITY, ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Beacon Shipping Lines,SELF_FUNDED,TPA 8
PLN032,395,4,6,389125.19,562,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Harbor Port Authority,SELF_FUNDED,TPA 3
PLN039,369,4,8,383826.23,521,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Orion Chemical Works,FULLY_INSURED,TPA 5
PLN033,351,4,7,360888.6,487,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Ironclad Security Services,FULLY_INSURED,TPA 2
PLN004,166,4,8,57456.56,167,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Delta Logistics,SELF_FUNDED,TPA 2
PLN001,165,3,7,44857.93,166,"List(ELIGIBILITY, ACCUMULATOR, ROLLOVER)",Alpha Corp,SELF_FUNDED,TPA 1
PLN002,149,4,8,41098.73,152,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Beta Industries,SELF_FUNDED,TPA 1
PLN005,194,4,8,35892.48,199,"List(ELIGIBILITY, ACCUMULATOR, COPAY_PROGRAM, ROLLOVER)",Epsilon Teachers Fund,FULLY_INSURED,TPA 3


[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:16:12.744704+00:00", "notebook": "NB_07", "section": "plan_rollup", "status": "SUCCESS", "row_count": 0, "detail": "plans=40", "duration_sec": 0.0}


## Severity Distribution

In [0]:
print("Findings by severity and category:")
display(
    df_registry
    .groupBy("severity", "issue_category")
    .agg(
        F.count("*").alias("count"),
        F.round(F.sum("dollar_exposure"), 2).alias("dollar_exposure"),
    )
    .orderBy(
        F.when(F.col("severity") == "CRITICAL", 1)
        .when(F.col("severity") == "HIGH", 2)
        .when(F.col("severity") == "MEDIUM", 3)
        .otherwise(4),
        "issue_category",
    )
)

Findings by severity and category:


severity,issue_category,count,dollar_exposure
CRITICAL,ACCUMULATOR,2049,741076.63
CRITICAL,ELIGIBILITY,500,0.0
HIGH,COPAY_PROGRAM,3908,3051581.93
HIGH,ELIGIBILITY,125,0.0
HIGH,FILE_QUALITY,21,0.0
HIGH,ROLLOVER,375,0.0
MEDIUM,ELIGIBILITY,200,0.0
MEDIUM,FILE_QUALITY,11,0.0
MEDIUM,ROLLOVER,40,365.4


## Executive Summary

In [0]:
total_findings = df_registry.count()
total_members_affected = df_member_findings.select("member_id").distinct().count()
total_exposure = df_registry.agg(F.sum("dollar_exposure")).collect()[0][0] or 0
critical_count = df_registry.filter(F.col("severity") == "CRITICAL").count()
multi_cat_members = df_member_rollup.filter(F.col("overlap_count") > 1).count()

print("=" * 65)
print("  CROSS-NOTEBOOK CONVERGENCE SUMMARY")
print("=" * 65)
print(f"  Total findings across all audits:      {total_findings}")
print(f"  Unique members affected:               {total_members_affected}")
print(f"  Total dollar exposure:                 ${total_exposure:,.2f}")
print(f"  CRITICAL severity findings:            {critical_count}")
print(f"  Members in multiple audit categories:  {multi_cat_members}")
print("=" * 65)
print()
print("  Output tables:")
print("    pbm_sim.findings_registry       - every finding, common schema")
print("    pbm_sim.findings_member_rollup  - member-level aggregation")
print("    pbm_sim.findings_plan_rollup    - plan-level aggregation")
print("=" * 65)

  CROSS-NOTEBOOK CONVERGENCE SUMMARY
  Total findings across all audits:      7229
  Unique members affected:               5944
  Total dollar exposure:                 $3,793,023.96
  CRITICAL severity findings:            2549
  Members in multiple audit categories:  176

  Output tables:
    pbm_sim.findings_registry       - every finding, common schema
    pbm_sim.findings_member_rollup  - member-level aggregation
    pbm_sim.findings_plan_rollup    - plan-level aggregation


## Investigation Notes

The convergence query serves a different purpose than the individual audit
notebooks. NB01 through NB05 each answer "what's wrong in this domain?"
This notebook answers "where do the domains intersect, and which members
or plans should the team look at first?"

The overlap analysis is particularly useful for root cause investigation.
If the same members keep appearing across ELIGIBILITY and ACCUMULATOR
findings, the root cause is likely in the enrollment feed that both
domains depend on. Fixing the feed fixes both categories at once.

The plan-level rollup feeds into account management conversations.
A plan with findings across four categories and $50K in exposure needs
a different conversation than a plan with a handful of effective date
mismatches. The rollup gives the account team the numbers they need
to prioritize.

The three output tables (findings_registry, findings_member_rollup,
findings_plan_rollup) are ready for the executive dashboard.

In [0]:
log_entry(NB, "convergence_complete", "SUCCESS",
          detail=f"{total_members_affected} members, ${total_exposure:,.2f} exposure, "
                  f"{critical_count} CRITICAL")
log_summary(NB)

[PIPELINE_LOG] {"run_id": "RUN_20261017_201332", "timestamp": "2026-10-17T20:16:43.163395+00:00", "notebook": "NB_07", "section": "convergence_complete", "status": "SUCCESS", "row_count": 0, "detail": "5944 members, $3,793,023.96 exposure, 2549 CRITICAL", "duration_sec": 0.0}

  PIPELINE LOG SUMMARY: NB_07 (run: RUN_20261017_201332)
  [+] load_source_tables: SUCCESS
  [+] nb01_findings: SUCCESS
  [+] nb02_findings: SUCCESS
  [+] nb03_findings: SUCCESS
  [+] nb04_findings: SUCCESS
  [+] nb05_findings: SUCCESS
  [+] findings_registry: SUCCESS
  [+] member_rollup: SUCCESS
  [+] overlap_analysis: SUCCESS
  [+] plan_rollup: SUCCESS
  [+] convergence_complete: SUCCESS

